# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I am using one refresh rule: pages move up the queue when they are stale, still visible, and showing weak CTR for their rank. The rule emits one reason code per row:

- `stale_visible_ctr_fix` = stale enough to deserve a refresh, visible enough to matter, and low CTR for the position.
- `stale_visible` = stale and visible, but CTR is not yet the main issue.
- `ctr_fix_candidate` = visible enough for CTR work, but staleness is not the driver.
- `quick_win` = some demand exists and the page is already in the visible range.
- `monitor` = nothing strong enough to move it ahead.

The two signal checks below are the only facts I am leaning on: staleness behind the refresh flag, and CTR-vs-position behind the CTR-fix logic.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
OUTPUT_DIR = Path("../outputs")
CSV_PATH = OUTPUT_DIR / "baseline_action_score.csv"
METRICS_PATH = OUTPUT_DIR / "baseline_action_score_metrics.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)


def bucket_table(frame: pd.DataFrame, bucket: pd.Series) -> pd.DataFrame:
    return (
        frame.assign(bucket=bucket)
        .groupby("bucket", dropna=False)
        .agg(
            n=("content_id", "size"),
            declining_rate=("is_declining_label", "mean"),
            mean_days_since_update=("days_since_last_update", "mean"),
            mean_impressions=("impressions_90d", "mean"),
            mean_ctr=("ctr", "mean"),
        )
        .reset_index()
    )


freshness_bucket = np.where(df["days_since_last_update"] > 90, "stale_90d_plus", "recent_90d_or_less")
freshness_table = bucket_table(df, pd.Series(freshness_bucket, index=df.index))
print("Signal check 1: staleness behind the refresh flag")
print(freshness_table.to_string(index=False))
print("Verdict: CONFIRMED — pages untouched for more than 90 days decline more often, so staleness is a real refresh signal.")

visible = df[df["avg_position"].between(0.01, 20, inclusive="both")].copy()
visible["ctr_bucket"] = np.where(visible["ctr"] < 0.15, "low_ctr", "higher_ctr")
ctr_table = bucket_table(visible, visible["ctr_bucket"])
print()
print("Signal check 2: CTR-vs-position behind the CTR-fix logic")
print(ctr_table.to_string(index=False))
print("Verdict: CONFIRMED — among visible pages, low-CTR rows decline more often than higher-CTR rows, so the CTR-fix idea is supported.")

Signal check 1: staleness behind the refresh flag
            bucket     n  declining_rate  mean_days_since_update  mean_impressions  mean_ctr
recent_90d_or_less 20655        0.512031               18.838199       4219.161317  0.604856
    stale_90d_plus  9345        0.608454              106.350562       7369.097057  0.302696
Verdict: CONFIRMED — pages untouched for more than 90 days decline more often, so staleness is a real refresh signal.

Signal check 2: CTR-vs-position behind the CTR-fix logic
    bucket     n  declining_rate  mean_days_since_update  mean_impressions  mean_ctr
higher_ctr  9704        0.549361               46.752061       9126.915808  1.327963
   low_ctr 10552        0.607752               43.927123       2954.366187  0.025829
Verdict: CONFIRMED — among visible pages, low-CTR rows decline more often than higher-CTR rows, so the CTR-fix idea is supported.


## 2. Build the ranked queue (writes the CSV)

I am scoring with simple point rules, then sorting highest score first. The notebook writes `work/outputs/baseline_action_score.csv` and a small JSON receipt with the score breakdown, base rate, and top-K precision.

In [2]:
stale_90 = (df["days_since_last_update"] > 90).astype(int)
visible_20 = df["avg_position"].between(0.01, 20, inclusive="both").astype(int)
visible_500 = (df["impressions_90d"] >= 500).astype(int)
low_ctr_visible = ((visible_20 == 1) & (df["ctr"] < 0.15)).astype(int)
quick_win = ((df["search_volume"].fillna(0) >= 100) & (df["impressions_90d"] >= 100) & (visible_20 == 1)).astype(int)
age_old = (df["content_age_days"] >= 180).astype(int)

# Transparent point score: stale + visible + weak CTR gets priority.
df["baseline_action_score"] = (
    3 * stale_90
    + 2 * visible_500
    + 2 * low_ctr_visible
    + 1 * quick_win
    + 1 * age_old
)


def reason_code(row: pd.Series) -> str:
    if row["days_since_last_update"] > 90 and row["impressions_90d"] >= 500 and row["ctr"] < 0.15 and 0 < row["avg_position"] <= 20:
        return "stale_visible_ctr_fix"
    if row["days_since_last_update"] > 90 and row["impressions_90d"] >= 500:
        return "stale_visible"
    if 0 < row["avg_position"] <= 20 and row["ctr"] < 0.15:
        return "ctr_fix_candidate"
    if row["search_volume"] >= 100 and row["impressions_90d"] >= 100 and 0 < row["avg_position"] <= 20:
        return "quick_win"
    if row["impressions_90d"] >= 500 and row["content_age_days"] >= 180:
        return "aging_visible"
    return "monitor"


def action_label(row: pd.Series) -> str:
    if row["days_since_last_update"] > 90 and row["impressions_90d"] >= 500:
        return "refresh"
    if 0 < row["avg_position"] <= 20 and row["ctr"] < 0.15:
        return "optimize_ctr"
    if row["search_volume"] >= 100 and row["impressions_90d"] >= 100 and 0 < row["avg_position"] <= 20:
        return "expand"
    if row["impressions_90d"] >= 500:
        return "monitor"
    return "deprioritize"


df["reason_code"] = df.apply(reason_code, axis=1)
df["action_label"] = df.apply(action_label, axis=1)

queue_df = (
    df.sort_values(
        ["baseline_action_score", "impressions_90d", "ctr", "days_since_last_update"],
        ascending=[False, False, True, False],
    )
    .reset_index(drop=True)
    .copy()
)
queue_df.insert(0, "rank", np.arange(1, len(queue_df) + 1))

output_columns = [
    "rank",
    "content_id",
    "client_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "search_volume",
    "content_age_days",
    "is_declining_label",
]
queue_df[output_columns].to_csv(CSV_PATH, index=False)

metrics = {
    "rows": int(len(queue_df)),
    "base_rate": float(queue_df["is_declining_label"].mean()),
    "precision_at_10": float(queue_df.head(10)["is_declining_label"].mean()),
    "precision_at_20": float(queue_df.head(20)["is_declining_label"].mean()),
    "score_counts_top": {str(k): int(v) for k, v in queue_df["baseline_action_score"].value_counts().sort_index(ascending=False).head(10).items()},
    "reason_code_counts_top": {str(k): int(v) for k, v in queue_df["reason_code"].value_counts().head(10).items()},
    "signal_thresholds": {
        "stale_days": 90,
        "visible_impressions_floor": 500,
        "ctr_floor": 0.15,
        "quick_win_min_search_volume": 100,
        "quick_win_min_impressions": 100,
        "visible_position_ceiling": 20,
    },
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2, sort_keys=True))

print(f"Wrote {CSV_PATH}")
print(f"Wrote {METRICS_PATH}")
print(f"Base rate: {queue_df['is_declining_label'].mean():.3f}")
print(f"Precision@10: {queue_df.head(10)['is_declining_label'].mean():.3f}")
print(f"Precision@20: {queue_df.head(20)['is_declining_label'].mean():.3f}")
print("Top score counts:")
print(queue_df["baseline_action_score"].value_counts().sort_index(ascending=False).head(10).to_string())

Wrote ../outputs/baseline_action_score.csv
Wrote ../outputs/baseline_action_score_metrics.json
Base rate: 0.542
Precision@10: 0.500
Precision@20: 0.700
Top score counts:
baseline_action_score
9      99
8    1065
7     629
6    5081
5    2612
4    2630
3    6084
2    6228
1    3377
0    2195


## 3. Top-10 review

I am reading the first ten rows as a skeptic: action, why it is ranked here, and the condition that would make the rule wrong.

In [3]:
top_10 = queue_df.head(10).copy()

for _, row in top_10.iterrows():
    why = (
        f"stale {int(row['days_since_last_update'])}d, {int(row['impressions_90d'])} impressions, "
        f"CTR {row['ctr']:.2f}, avg position {row['avg_position']:.1f}"
    )
    wrong = (
        "wrong if the page was already refreshed, the metrics are stale, or the page got a temporary traffic spike"
    )
    print(f"{int(row['rank']):>2}. {row['action_label']} | {row['reason_code']} | why: {why} | what would make it wrong: {wrong}")

 1. refresh | stale_visible_ctr_fix | why: stale 104d, 517715 impressions, CTR 0.14, avg position 4.2 | what would make it wrong: wrong if the page was already refreshed, the metrics are stale, or the page got a temporary traffic spike
 2. refresh | stale_visible_ctr_fix | why: stale 104d, 97235 impressions, CTR 0.07, avg position 6.5 | what would make it wrong: wrong if the page was already refreshed, the metrics are stale, or the page got a temporary traffic spike
 3. refresh | stale_visible_ctr_fix | why: stale 104d, 77612 impressions, CTR 0.10, avg position 7.1 | what would make it wrong: wrong if the page was already refreshed, the metrics are stale, or the page got a temporary traffic spike
 4. refresh | stale_visible_ctr_fix | why: stale 104d, 66359 impressions, CTR 0.04, avg position 4.9 | what would make it wrong: wrong if the page was already refreshed, the metrics are stale, or the page got a temporary traffic spike
 5. refresh | stale_visible_ctr_fix | why: stale 104d, 6336

## 4. Weak picks + leakage check

I am calling out the weakest rows in the top 20, then checking that the score uses only past-looking inputs: no `trend_pct`, no `trend_direction`, no label-derived fields, and no future-window columns.

In [4]:
top_20 = queue_df.head(20).copy()

weak_picks = top_20.sort_values(["baseline_action_score", "impressions_90d", "ctr"], ascending=[True, True, False]).head(3)
print("Weak picks inside the top 20:")
for _, row in weak_picks.iterrows():
    note = (
        f"score {int(row['baseline_action_score'])}, impressions {int(row['impressions_90d'])}, "
        f"CTR {row['ctr']:.2f}, avg position {row['avg_position']:.1f}"
    )
    wrong = (
        "wrong if a manual refresh already happened, if the CTR is noisy because the page is too small, "
        "or if the visibility is not stable enough to justify work"
    )
    print(f"{int(row['rank']):>2}. {row['action_label']} | {row['reason_code']} | why weaker: {note} | what would make it wrong: {wrong}")

forbidden_inputs = {"trend_pct", "trend_direction", "is_declining_label"}
score_inputs = {
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "search_volume",
    "content_age_days",
}

print()
print("Leakage check:")
print("forbidden inputs used in score:", sorted(forbidden_inputs & score_inputs))
print("score inputs:", sorted(score_inputs))
print("pass: the score uses only past-looking inputs and does not read the label or trend_pct.")

Weak picks inside the top 20:
20. refresh | stale_visible_ctr_fix | why weaker: score 9, impressions 10003, CTR 0.06, avg position 8.6 | what would make it wrong: wrong if a manual refresh already happened, if the CTR is noisy because the page is too small, or if the visibility is not stable enough to justify work
19. refresh | stale_visible_ctr_fix | why weaker: score 9, impressions 10210, CTR 0.12, avg position 9.6 | what would make it wrong: wrong if a manual refresh already happened, if the CTR is noisy because the page is too small, or if the visibility is not stable enough to justify work
18. refresh | stale_visible_ctr_fix | why weaker: score 9, impressions 10339, CTR 0.08, avg position 10.9 | what would make it wrong: wrong if a manual refresh already happened, if the CTR is noisy because the page is too small, or if the visibility is not stable enough to justify work

Leakage check:
forbidden inputs used in score: []
score inputs: ['avg_position', 'content_age_days', 'ctr', 'd

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.